# HW04: Conditional Generative Models for Medical Image Synthesis

**Course**: CSYE 7374 — Deep Learning and Generative AI in Healthcare

---

## Objectives

In this homework you will implement two conditional generative models and compare their behaviour on retinal OCT scans:

1. **Conditional GAN (cGAN)** — implement the generator and discriminator forward passes and the adversarial training step
2. **Denoising Diffusion Probabilistic Model (DDPM)** — implement the forward diffusion process and the noise-prediction training objective
3. **Comparison** — visualize generated images from both models side-by-side and analyse the differences

---

## Dataset: OCTMNIST

**OCTMNIST** contains 109,309 grayscale retinal OCT (Optical Coherence Tomography) scans categorised into **4 classes**:

| Label | Class | Description |
|-------|-------|-------------|
| 0 | CNV | Choroidal neovascularisation |
| 1 | DME | Diabetic macular edema |
| 2 | Drusen | Drusen deposits |
| 3 | Normal | Healthy retina |

Images are 28 x 28 pixels, single channel.

---

## Instructions

- Complete all cells marked with **`# TODO`**
- Do **not** modify provided helper functions or model architectures unless instructed
- Run all cells in order before submitting
- Answer the four analysis questions at the end in the provided markdown cells

---

## Grading Rubric

| Task | Points |
|------|--------|
| Data loading (val / test splits) | 5 |
| Sample visualisation per class | 5 |
| cGAN: `Generator.forward()` | 15 |
| cGAN: `Discriminator.forward()` | 10 |
| cGAN: Training step (D loss + G loss) | 15 |
| DDPM: `q_sample` -- forward diffusion | 10 |
| DDPM: Training step (noise prediction + loss) | 10 |
| Generated image visualisation (both models) | 10 |
| Analysis questions (4 x 5 points) | 20 |
| **Total** | **100** |


---
## 1. Setup and Imports

In [ ]:
!pip install -q medmnist torch torchvision matplotlib seaborn tqdm numpy

In [ ]:
import math, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from tqdm.notebook import tqdm
import medmnist
from medmnist import INFO

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
print(f'PyTorch {torch.__version__}  |  MedMNIST {medmnist.__version__}')


---
## 2. Configuration

In [ ]:
class Config:
    DATA_FLAG    = 'octmnist'
    IMG_SIZE     = 28
    N_CH         = 1
    N_CLASSES    = 4
    BATCH_SIZE   = 256
    SEED         = 42
    # cGAN
    LATENT_DIM   = 64
    GAN_LR       = 2e-4
    GAN_EPOCHS   = 60
    # DDPM
    T            = 400
    BETA_START   = 1e-4
    BETA_END     = 0.02
    DDPM_LR      = 5e-4
    DDPM_EPOCHS  = 60
    TIME_DIM     = 128

device = (torch.device('cuda')  if torch.cuda.is_available() else
          torch.device('mps')   if torch.backends.mps.is_available() else
          torch.device('cpu'))
torch.manual_seed(Config.SEED)
np.random.seed(Config.SEED)
print(f'Device: {device}')

CLASS_NAMES = list(INFO[Config.DATA_FLAG]['label'].values())
print(f'Classes: {CLASS_NAMES}')


---
## 3. Data Loading and Exploration

In [ ]:
DataClass = getattr(medmnist, INFO[Config.DATA_FLAG]['python_class'])
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])   # scale to [-1, 1]
])

train_ds = DataClass(split='train', transform=tfm, download=True)
print(f'Training set : {len(train_ds)} samples')
print(f'Image shape  : {train_ds[0][0].shape}')
print(f'Classes      : {CLASS_NAMES}')


In [ ]:
# ============================================================
# TODO: Load the validation and test splits.
# Use the same transform (tfm) and download=True.
# ============================================================
val_ds  = None  # TODO
test_ds = None  # TODO
# ============================================================

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')


In [ ]:
# num_workers=0 ensures compatibility on Windows and Colab
train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE,
                          shuffle=True,  num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=Config.BATCH_SIZE,
                          shuffle=False, num_workers=0)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


In [ ]:
# ============================================================
# TODO: Visualize 2 sample images from each class.
#
# Create a figure with 2 rows and N_CLASSES columns.
# For each class:
#   - Find 2 image indices in train_ds where the label equals that class
#   - Display each image using imshow(..., cmap='gray', vmin=-1, vmax=1)
#   - Set the column title (row 0 only) to CLASS_NAMES[cls]
#   - Turn off axis ticks for every subplot
# ============================================================

fig, axes = plt.subplots(2, Config.N_CLASSES, figsize=(Config.N_CLASSES * 2.5, 5))

# TODO: Fill in the visualisation

plt.suptitle('OCTMNIST -- Sample Images per Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
# ============================================================


---
## 4. Part 1 -- Conditional GAN (cGAN)

### Architecture and Loss

**Generator** $G(z, c)$: concatenates noise $z \in \mathbb{R}^{64}$ with a class embedding, then upsamples to a 28x28 OCT image.

**Discriminator** $D(x, c)$: concatenates a spatial class map to the image channels, then outputs a real/fake logit.

**Loss**:
$$\mathcal{L}_D = \tfrac{1}{2}\!\left[-\mathbb{E}[\log D(x,c)] - \mathbb{E}[\log(1-D(G(z,c),c))]\right]$$
$$\mathcal{L}_G = -\mathbb{E}[\log D(G(z,c),c)]$$


In [ ]:
class Generator(nn.Module):
    """
    Conditional GAN Generator.

    Input  : latent noise z (B, LATENT_DIM) + class label c (B,)
    Output : generated image (B, N_CH, IMG_SIZE, IMG_SIZE) in [-1, 1]

    Architecture:
        class_emb(c)  ->  16-dim vector
        concat(z, class_emb)  ->  fc  ->  reshape to (B, 256, 7, 7)
        ConvTranspose 256->128  (7->14)
        ConvTranspose 128->64  (14->28)
        Conv 64->N_CH  ->  Tanh
    """
    def __init__(self):
        super().__init__()
        self.class_emb = nn.Embedding(Config.N_CLASSES, 16)
        self.fc = nn.Sequential(
            nn.Linear(Config.LATENT_DIM + 16, 256 * 7 * 7),
            nn.ReLU(True)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128,  64, 4, 2, 1), nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.Conv2d(64, Config.N_CH, 3, 1, 1),  nn.Tanh()
        )

    def forward(self, z, c):
        """
        Args:
            z : latent noise,   shape (B, LATENT_DIM)
            c : class labels,   shape (B,)  -- integer LongTensor
        Returns:
            generated images,   shape (B, N_CH, IMG_SIZE, IMG_SIZE)

        Steps:
          1. Look up the class embedding: emb = self.class_emb(c)   -> (B, 16)
          2. Concatenate z and emb along dim=1                       -> (B, LATENT_DIM+16)
          3. Pass through self.fc and reshape to (B, 256, 7, 7)
          4. Pass through self.decoder and return
        """
        # ===========================================================
        # TODO: Implement the forward pass  (~4 lines)
        # ===========================================================

        pass  # remove this line after implementing

        # ===========================================================


In [ ]:
class Discriminator(nn.Module):
    """
    Conditional GAN Discriminator.

    Input  : image (B, N_CH, IMG_SIZE, IMG_SIZE) + class label c (B,)
    Output : real/fake logit  (B, 1)

    Architecture:
        class_emb(c)  ->  reshape to (B, 1, IMG_SIZE, IMG_SIZE)  -- spatial class map
        concat(img, class_map) along channels  ->  (B, N_CH+1, 28, 28)
        Conv  2->64   (28->14)
        Conv 64->128  (14->7)
        Flatten  ->  Linear  ->  logit
    """
    def __init__(self):
        super().__init__()
        self.class_emb = nn.Embedding(Config.N_CLASSES, Config.IMG_SIZE ** 2)
        self.net = nn.Sequential(
            nn.Conv2d(Config.N_CH + 1, 64,  4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64,             128,  4, 2, 1), nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, True),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1)
        )

    def forward(self, img, c):
        """
        Args:
            img : images,       shape (B, N_CH, IMG_SIZE, IMG_SIZE)
            c   : class labels, shape (B,)  -- integer LongTensor
        Returns:
            real/fake logit,    shape (B, 1)

        Steps:
          1. Look up the class embedding: emb = self.class_emb(c)        -> (B, IMG_SIZE^2)
          2. Reshape emb to a spatial map: c_map shape (B, 1, IMG_SIZE, IMG_SIZE)
          3. Concatenate img and c_map along dim=1                        -> (B, N_CH+1, 28, 28)
          4. Pass through self.net and return
        """
        # ===========================================================
        # TODO: Implement the forward pass  (~3 lines)
        # ===========================================================

        pass  # remove this line after implementing

        # ===========================================================


In [ ]:
G = Generator().to(device)
D = Discriminator().to(device)
print(f'Generator     params: {sum(p.numel() for p in G.parameters()):,}')
print(f'Discriminator params: {sum(p.numel() for p in D.parameters()):,}')

# Smoke-test: passes once both forward() methods are implemented
try:
    _z     = torch.randn(4, Config.LATENT_DIM, device=device)
    _c     = torch.zeros(4, dtype=torch.long, device=device)
    _out   = G(_z, _c)
    _logit = D(_out, _c)
    print(f'Generator output shape    : {_out.shape}')    # expect (4, 1, 28, 28)
    print(f'Discriminator output shape: {_logit.shape}')  # expect (4, 1)
    print('Smoke-test passed.')
except Exception as e:
    print(f'Smoke-test failed (complete the TODOs first): {e}')


In [ ]:
opt_G = optim.Adam(G.parameters(), lr=Config.GAN_LR, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=Config.GAN_LR, betas=(0.5, 0.999))
bce   = nn.BCEWithLogitsLoss()
print('Optimizers and BCE loss ready.')


### cGAN Training

In [ ]:
def train_cgan_epoch(G, D, loader, opt_G, opt_D, bce, device):
    """Train cGAN for one epoch. Returns (avg_d_loss, avg_g_loss)."""
    G.train(); D.train()
    d_sum = g_sum = 0.0

    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.squeeze().long().to(device)
        B      = imgs.size(0)
        real   = torch.ones(B,  1, device=device)
        fake   = torch.zeros(B, 1, device=device)

        # ============================================================
        # TODO: Train the Discriminator
        #
        # Steps:
        #   1. Sample z ~ N(0, I) of shape (B, Config.LATENT_DIM)
        #   2. Generate fake images: fake_imgs = G(z, labels)
        #   3. Compute the discriminator loss:
        #        d_real = bce(D(imgs,               labels), real)
        #        d_fake = bce(D(fake_imgs.detach(), labels), fake)
        #        loss_D = 0.5 * (d_real + d_fake)
        #   4. opt_D.zero_grad()  ->  loss_D.backward()  ->  opt_D.step()
        # ============================================================

        loss_D = None  # TODO

        # ============================================================
        # TODO: Train the Generator
        #
        # Steps:
        #   1. Sample a fresh z ~ N(0, I) of shape (B, Config.LATENT_DIM)
        #   2. Generate images: gen_imgs = G(z, labels)
        #   3. The generator wants D to label these as real:
        #        loss_G = bce(D(gen_imgs, labels), real)
        #   4. opt_G.zero_grad()  ->  loss_G.backward()  ->  opt_G.step()
        # ============================================================

        loss_G = None  # TODO

        d_sum += loss_D.item()
        g_sum += loss_G.item()

    n = len(loader)
    return d_sum / n, g_sum / n


In [ ]:
gan_history = {'d': [], 'g': []}

for epoch in range(1, Config.GAN_EPOCHS + 1):
    d_loss, g_loss = train_cgan_epoch(G, D, train_loader, opt_G, opt_D, bce, device)
    gan_history['d'].append(d_loss)
    gan_history['g'].append(g_loss)
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{Config.GAN_EPOCHS} | D: {d_loss:.4f}  G: {g_loss:.4f}')


In [ ]:
G.eval()

# ============================================================
# TODO: Visualize 2 cGAN-generated samples per class.
#
# For each class cls in range(N_CLASSES):
#   1. z   = torch.randn(2, Config.LATENT_DIM, device=device)
#   2. lbl = torch.full((2,), cls, dtype=torch.long, device=device)
#   3. with torch.no_grad(): out = G(z, lbl)
#   4. Plot each generated image: cmap='gray', vmin=-1, vmax=1
#   5. Set column title (row 0) to CLASS_NAMES[cls]; turn off axis
# ============================================================

fig, axes = plt.subplots(2, Config.N_CLASSES, figsize=(Config.N_CLASSES * 2.5, 5))

# TODO: Fill in the visualisation

plt.suptitle('cGAN -- Generated Samples (2 per class)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
# ============================================================


---
## 5. Part 2 -- Denoising Diffusion Probabilistic Model (DDPM)

### Forward Diffusion and Training Objective

**Forward process** (closed-form corruption at any timestep $t$):
$$q(x_t \mid x_0) = \mathcal{N}\!\left(\sqrt{\bar{\alpha}_t}\, x_0,\; (1-\bar{\alpha}_t)\mathbf{I}\right) \implies x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \mathbf{I})$$

**Training objective** -- predict the added noise given noisy image $x_t$, timestep $t$, and class $c$:
$$\mathcal{L} = \mathbb{E}_{x_0,\,\varepsilon,\,t}\!\left[\|\varepsilon - \varepsilon_\theta(x_t, t, c)\|^2\right]$$

**Reverse sampling** (provided -- you do not need to implement this):
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\,\varepsilon_\theta(x_t,t,c)\right) + \sqrt{\beta_t}\,z, \quad z \sim \mathcal{N}(0,\mathbf{I})$$


In [ ]:
# Noise schedule (provided -- do not modify)
betas     = torch.linspace(Config.BETA_START, Config.BETA_END, Config.T, device=device)
alphas    = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)   # alpha_bar_t
sqrt_ab   = alpha_bar.sqrt()               # sqrt(alpha_bar_t)
sqrt_1mab = (1.0 - alpha_bar).sqrt()       # sqrt(1 - alpha_bar_t)

def extract(a, t, shape):
    """Gather scalar values at indices t and broadcast to the image shape."""
    v = a.gather(0, t)
    return v.reshape(t.shape[0], *((1,) * (len(shape) - 1)))

print(f'Noise schedule ready: T={Config.T}, beta in [{betas[0]:.4f}, {betas[-1]:.4f}]')


In [ ]:
def q_sample(x0, t, noise):
    """
    Forward diffusion: corrupt x0 to x_t in a single step.

    Closed-form formula:
        x_t = sqrt(alpha_bar_t) * x0  +  sqrt(1 - alpha_bar_t) * noise

    Args:
        x0    : clean image,            shape (B, C, H, W)
        t     : timestep indices,       shape (B,)  -- LongTensor in [0, T)
        noise : Gaussian noise,         same shape as x0
    Returns:
        x_t   : noisy image at step t,  same shape as x0

    Hint: use extract(sqrt_ab, t, x0.shape) to broadcast the scalar
          coefficient to the correct shape for element-wise multiplication.
    """
    # ===========================================================
    # TODO: Implement q_sample  (~1 line)
    # ===========================================================

    pass  # remove this line after implementing

    # ===========================================================

# Sanity check
try:
    _x0    = torch.randn(4, Config.N_CH, Config.IMG_SIZE, Config.IMG_SIZE, device=device)
    _t     = torch.randint(0, Config.T, (4,), device=device, dtype=torch.long)
    _noise = torch.randn_like(_x0)
    _xt    = q_sample(_x0, _t, _noise)
    assert _xt.shape == _x0.shape, f"Shape mismatch: {_xt.shape}"
    print("q_sample sanity check passed.")
except Exception as e:
    print(f"q_sample check failed (complete the TODO first): {e}")


In [ ]:
# Conditional U-Net for DDPM (provided -- do not modify)

class SinEmb(nn.Module):
    """Sinusoidal timestep embedding."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freq = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        emb  = t[:, None].float() * freq[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class DBlock(nn.Module):
    """Conv block conditioned on time and class embeddings."""
    def __init__(self, in_ch, out_ch, tdim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1)
        self.gn1   = nn.GroupNorm(8, out_ch)
        self.gn2   = nn.GroupNorm(8, out_ch)
        self.temb  = nn.Linear(tdim, out_ch)
        self.cemb  = nn.Linear(tdim, out_ch)
    def forward(self, x, te, ce):
        h = F.silu(self.gn1(self.conv1(x)))
        h = h + self.temb(te)[:, :, None, None] + self.cemb(ce)[:, :, None, None]
        return F.silu(self.gn2(self.conv2(h)))


class UNetDDPM(nn.Module):
    """Conditional U-Net denoising network for 28x28 single-channel images."""
    def __init__(self):
        super().__init__()
        td = Config.TIME_DIM
        self.time_mlp  = nn.Sequential(SinEmb(td), nn.Linear(td, td), nn.SiLU())
        self.class_emb = nn.Embedding(Config.N_CLASSES, td)
        self.e1   = DBlock(Config.N_CH, 64,  td)      # 28x28
        self.e2   = DBlock(64,  128, td)               # 14x14
        self.bn   = DBlock(128, 256, td)               #  7x7
        self.d1   = DBlock(256 + 128, 128, td)         # 14x14
        self.d2   = DBlock(128 +  64,  64, td)         # 28x28
        self.out  = nn.Conv2d(64, Config.N_CH, 1)
        self.pool = nn.MaxPool2d(2)
        self.up1  = nn.ConvTranspose2d(256, 256, 2, 2)
        self.up2  = nn.ConvTranspose2d(128, 128, 2, 2)

    def forward(self, x, t, c):
        te = self.time_mlp(t)
        ce = self.class_emb(c)
        e1 = self.e1(x,              te, ce)
        e2 = self.e2(self.pool(e1),  te, ce)
        b  = self.bn(self.pool(e2),  te, ce)
        d1 = self.d1(torch.cat([self.up1(b),  e2], 1), te, ce)
        d2 = self.d2(torch.cat([self.up2(d1), e1], 1), te, ce)
        return self.out(d2)


unet = UNetDDPM().to(device)
print(f"UNet params: {sum(p.numel() for p in unet.parameters()):,}")


In [ ]:
opt_ddpm     = optim.Adam(unet.parameters(), lr=Config.DDPM_LR)
ddpm_history = []

for epoch in range(1, Config.DDPM_EPOCHS + 1):
    unet.train()
    ep_loss = 0.0

    for imgs, labels in train_loader:
        imgs   = imgs.to(device)
        labels = labels.squeeze().long().to(device)

        # ============================================================
        # TODO: Complete the DDPM training step
        #
        # Steps:
        #   1. Sample a random timestep for each image:
        #        t = torch.randint(0, Config.T, (imgs.size(0),),
        #                          device=device, dtype=torch.long)
        #   2. Sample Gaussian noise matching the image shape:
        #        noise = torch.randn_like(imgs)
        #   3. Corrupt the images using your q_sample function:
        #        x_t = q_sample(imgs, t, noise)
        #   4. Predict the noise with the U-Net:
        #        pred = unet(x_t, t, labels)
        #   5. Compute MSE loss between the prediction and the true noise:
        #        loss = F.mse_loss(pred, noise)
        #   6. opt_ddpm.zero_grad()  ->  loss.backward()  ->  opt_ddpm.step()
        # ============================================================

        loss = None  # TODO

        # ============================================================

        ep_loss += loss.item()

    ddpm_history.append(ep_loss / len(train_loader))
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{Config.DDPM_EPOCHS} | Loss: {ddpm_history[-1]:.5f}')


In [ ]:
@torch.no_grad()
def ddpm_sample(class_list):
    """
    Generate images via the full DDPM reverse process (provided -- do not modify).

    Args:
        class_list : list of integer class labels, one per sample to generate
    Returns:
        Tensor of generated images shape (len(class_list), N_CH, IMG_SIZE, IMG_SIZE),
        values clamped to [-1, 1]
    """
    unet.eval()
    n = len(class_list)
    x = torch.randn(n, Config.N_CH, Config.IMG_SIZE, Config.IMG_SIZE, device=device)
    c = torch.tensor(class_list, dtype=torch.long, device=device)
    for i in tqdm(reversed(range(Config.T)), total=Config.T, desc="Sampling", leave=False):
        t    = torch.full((n,), i, device=device, dtype=torch.long)
        eps  = unet(x, t, c)
        b_t  = betas[i]; a_t = alphas[i]; ab_t = alpha_bar[i]
        mean = (1.0 / a_t.sqrt()) * (x - b_t / (1.0 - ab_t).sqrt() * eps)
        x    = mean if i == 0 else mean + b_t.sqrt() * torch.randn_like(x)
    return x.clamp(-1, 1)


In [ ]:
# ============================================================
# TODO: Generate and visualize 2 DDPM samples per class.
#
# Steps:
#   1. Call ddpm_sample with class list [0, 0, 1, 1, 2, 2, 3, 3]
#      (2 samples for each of the 4 classes, 8 total)
#   2. Create a 2-row x N_CLASSES column figure
#   3. For each class index cls:
#        samples[cls*2]     -> row 0, column cls
#        samples[cls*2 + 1] -> row 1, column cls
#   4. Display with cmap='gray', vmin=-1, vmax=1
#   5. Set column title (row 0) to CLASS_NAMES[cls]; turn off axis
# ============================================================

# TODO: Call ddpm_sample here

fig, axes = plt.subplots(2, Config.N_CLASSES, figsize=(Config.N_CLASSES * 2.5, 5))

# TODO: Fill in the visualisation

plt.suptitle('DDPM -- Generated Samples (2 per class)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
# ============================================================


---
## 6. Comparison

In [ ]:
# Training dynamics comparison (provided)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(gan_history['d'], label='Discriminator')
axes[0].plot(gan_history['g'], label='Generator')
axes[0].set_title('cGAN Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(ddpm_history, color='steelblue', label='Noise MSE')
axes[1].set_title('DDPM Training Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Training Dynamics Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Side-by-side: Real | GAN (x2) | DDPM (x2) for each class (provided)
G.eval()
show_cls = list(range(Config.N_CLASSES))
N = len(show_cls)

with torch.no_grad():
    lbl_t   = torch.tensor([c for c in show_cls for _ in range(2)],
                             dtype=torch.long, device=device)
    z       = torch.randn(N * 2, Config.LATENT_DIM, device=device)
    gan_out = G(z, lbl_t).cpu()

ddpm_out = ddpm_sample([c for c in show_cls for _ in range(2)]).cpu()

fig, axes = plt.subplots(N, 5, figsize=(11, N * 2.4))
for col, title in enumerate(['Real', 'GAN (1)', 'GAN (2)', 'DDPM (1)', 'DDPM (2)']):
    axes[0, col].set_title(title, fontsize=10, fontweight='bold')

for row, cls in enumerate(show_cls):
    real_idx = next(i for i, (_, l) in enumerate(train_ds) if int(l) == cls)
    real, _  = train_ds[real_idx]
    srcs = [real,
            gan_out[row*2],   gan_out[row*2 + 1],
            ddpm_out[row*2],  ddpm_out[row*2 + 1]]
    axes[row, 0].set_ylabel(CLASS_NAMES[cls], fontsize=9)
    for col, img in enumerate(srcs):
        axes[row, col].imshow(img.squeeze(), cmap='gray', vmin=-1, vmax=1)
        axes[row, col].axis('off')

plt.suptitle('Real vs cGAN vs DDPM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 7. Analysis Questions (20 points)

Answer each question in the markdown cell immediately below it.

### Question 1 (5 points) -- Training Stability

Look at the training loss curves for both models. The cGAN has two losses (D and G) while the DDPM has a single MSE loss.

1. Describe the typical shape and behaviour of each loss curve over epochs.
2. Why is cGAN training harder to monitor and diagnose than DDPM training?
3. Name one practical symptom of cGAN training failure (e.g. mode collapse or vanishing gradients for the generator) and explain how you would detect it from the loss curves or from the generated images.


**Your Answer:**

*[Write your answer here]*

### Question 2 (5 points) -- Output Quality

Examine the side-by-side comparison grid (Real vs cGAN vs DDPM).

1. Describe one visual difference between cGAN-generated and DDPM-generated images for at least two of the four classes.
2. For each model, suggest one specific change (architectural or to the training procedure) that could improve the visual quality of its generated OCT scans.
3. Which model would you prefer for a clinical data-augmentation task requiring anatomically realistic images, and why?


**Your Answer:**

*[Write your answer here]*

### Question 3 (5 points) -- Conditioning Mechanism

Both models receive the class label $c$ as a conditioning signal, but inject it differently.

1. In the **Generator**, where and how is $c$ combined with the latent noise $z$? (Be specific about the operation and where in the network it happens.)
2. In the **UNetDDPM**, where and how is $c$ injected, and at how many locations in the forward pass?
3. Which injection strategy do you think provides stronger class conditioning, and why?


**Your Answer:**

*[Write your answer here]*

### Question 4 (5 points) -- Clinical Application

You are a researcher who needs to augment a training set for **Drusen detection** (class 2). You have only 80 real Drusen scans and want to generate 500 additional synthetic ones.

1. Would you choose cGAN or DDPM? Give at least two reasons considering training requirements, inference speed, and sample diversity.
2. Beyond visual inspection, what quantitative metric would you use to verify the synthetic images are useful for downstream classifier training? Briefly explain how it works.
3. Identify one risk of using synthetic data for clinical model training and describe how you would mitigate it.


**Your Answer:**

*[Write your answer here]*

---
## Submission Instructions

1. Run all cells top-to-bottom and confirm there are **no errors**
2. Ensure all plots are rendered and every `# TODO` block is replaced with working code
3. Rename this notebook: **`FirstName_LastName_HW04.ipynb`**
4. Submit via the course portal by the posted deadline

**Example filename:** `Jane_Smith_HW04.ipynb`

**Your submission must show:**
- All TODOs completed with working code
- Smoke-test and sanity-check cells printing success messages
- Training output printed at every 10th epoch for both models
- All visualisation plots rendered (samples per class, cGAN grid, DDPM grid, training curves, comparison grid)
- Analysis questions answered in complete sentences
